# Weekly Project: Image Classification with Transfer Learning

In this project, you will build a complete image classification pipeline using transfer learning. You'll work with the **Intel Image Classification** dataset, which contains images of natural scenes across 6 categories: buildings, forest, glacier, mountain, sea, and street.

**Learning Objectives:**
- Load and prepare image datasets for deep learning
- Use pre-trained models for transfer learning
- Implement two transfer learning strategies: fine-tuning and feature extraction
- Evaluate model performance
- Deploy models using ONNX for production

**Dataset:** [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification/data)

**References:**

- [Training with PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)
- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

## Table of Contents

1. [Data Ingestion](#1)
2. [Data Preparation](#2)
3. [Model Building](#3)
4. [Training](#4)
   - [4.1 ConvNet as Fixed Feature Extractor](#4-1)
   - [4.2 Fine-tuning the ConvNet](#4-2)
5. [Evaluation](#5)
6. [Inference on Custom Images](#6)
7. [Deployment (ONNX)](#7)

## Imports

In [ ]:
%matplotlib inline

import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
from PIL import Image
import onnx
import onnxruntime as ort

import helper_utils

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

<a name='1'></a>
## 1. Data Ingestion

The Intel Image Classification dataset contains images organized into 6 classes. The dataset should be downloaded from [Kaggle](https://www.kaggle.com/datasets/puneet6060/intel-image-classification/data) and extracted to a local directory.

**Task:** Download the dataset and load it using `torchvision.datasets.ImageFolder`. The dataset structure should be:
```
data/
  seg_train/
    buildings/
    forest/
    glacier/
    mountain/
    sea/
    street/
  seg_test/
    buildings/
    forest/
    ...
```

**References:**

- [Dataset and DataLoader](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html#dataset-and-dataloader)
- [torchvision.datasets.ImageFolder](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html)

In [ ]:
# TODO: Download the Intel Image Classification dataset from Kaggle
# Extract it to a directory (e.g., './data/intel-image-classification')

data_dir = './data/intel-image-classification'

# Load datasets (without transforms for now)
# Use ImageFolder to load train and test sets
# The train set is in 'seg_train' and test set is in 'seg_test'

# YOUR CODE HERE
# train_dataset = datasets.ImageFolder(...)
# test_dataset = datasets.ImageFolder(...)

# Get class names
# class_names = train_dataset.classes
# print(f"Classes: {class_names}")
# print(f"Training samples: {len(train_dataset)}")
# print(f"Test samples: {len(test_dataset)}")

<a name='2'></a>
## 2. Data Preparation

Before training, we need to:
1. Define data transformations (augmentation for training, normalization for both)
2. Create DataLoaders for efficient batch processing

**Task:** Create transformation pipelines for training and validation. Pre-trained models expect ImageNet normalization statistics.

**Reference:** 
- [torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)

In [ ]:
# Define data transformations
# ImageNet normalization: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]

# YOUR CODE HERE
# data_transforms = {
#     'train': transforms.Compose([
#         # Add data augmentation transforms
#         # transforms.RandomResizedCrop(224),
#         # transforms.RandomHorizontalFlip(),
#         # transforms.ToTensor(),
#         # transforms.Normalize(...)
#     ]),
#     'val': transforms.Compose([
#         # Add validation transforms (no augmentation)
#         # transforms.Resize(256),
#         # transforms.CenterCrop(224),
#         # transforms.ToTensor(),
#         # transforms.Normalize(...)
#     ]),
# }

In [ ]:
# Create datasets with transformations
# YOUR CODE HERE
# image_datasets = {
#     'train': datasets.ImageFolder(..., transform=data_transforms['train']),
#     'val': datasets.ImageFolder(..., transform=data_transforms['val'])
# }

# Create data loaders
# YOUR CODE HERE
# dataloaders = {
#     'train': DataLoader(..., batch_size=32, shuffle=True, num_workers=4),
#     'val': DataLoader(..., batch_size=32, shuffle=False, num_workers=4)
# }

# dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
# class_names = image_datasets['train'].classes

# print(f"Training samples: {dataset_sizes['train']}")
# print(f"Validation samples: {dataset_sizes['val']}")
# print(f"Classes: {class_names}")

In [ ]:
# Visualize a batch of training images
# helper_utils.visualize_batch(dataloaders['train'], class_names, num_images=8)

<a name='3'></a>
## 3. Model Building

We'll use a pre-trained ResNet-18 model and adapt it for our 6-class classification task.

**Task:** Load a pre-trained ResNet-18 model and modify the final layer for 6 classes.

**Reference:** 

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision.models](https://pytorch.org/vision/stable/models.html)
- [ResNet documentation](https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)

In [ ]:
# Load pre-trained ResNet-18
# YOUR CODE HERE
# model = models.resnet18(weights='IMAGENET1K_V1')
# model = model.to(device)

# Modify the final layer for 6 classes
# num_ftrs = model.fc.in_features
# model.fc = nn.Linear(num_ftrs, len(class_names))

# print(f"Model loaded. Final layer: {model.fc}")

<a name='4'></a>
## 4. Training

We'll implement a training function and then train using two different transfer learning strategies.

**Task:** Implement a training function that:
- Handles both training and validation phases
- Tracks loss and accuracy
- Saves the best model based on validation accuracy
- Uses a learning rate scheduler

**Reference:** [PyTorch Training Tutorial](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html#the-training-loop)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    """
    Train a model with training and validation phases.
    
    Returns:
        model: Trained model with best weights loaded
        history: Dictionary with training history
    """
    since = time.time()
    
    # Initialize history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            # Store history
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it's the best so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

<a name='4-1'></a>
### 4.1 ConvNet as Fixed Feature Extractor

In this approach, we freeze all the convolutional layers and only train the final classifier layer.

**Task:** 
1. Load a fresh pre-trained model
2. Freeze all parameters except the final layer
3. Set up optimizer to only train the final layer
4. Train the model

**Reference:** [Freezing Parameters](https://pytorch.org/docs/notes/autograd.html#excluding-subgraphs-from-backward)

In [ ]:
# Load a fresh pre-trained model
# YOUR CODE HERE
# model_conv = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all parameters
# for param in model_conv.parameters():
#     param.requires_grad = False

# Modify final layer
# num_ftrs = model_conv.fc.in_features
# model_conv.fc = nn.Linear(num_ftrs, len(class_names))
# model_conv = model_conv.to(device)

# Set up loss function and optimizer (only for final layer)
# criterion = nn.CrossEntropyLoss()
# optimizer_conv = optim.SGD(model_conv.fc.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
# exp_lr_scheduler = lr_scheduler.StepLR(optimizer_conv, step_size=7, gamma=0.1)

# Train the model
# model_conv, history_conv = train_model(model_conv, criterion, optimizer_conv, exp_lr_scheduler, num_epochs=10)

In [ ]:
# Visualize training history
# helper_utils.visualize_training_history(history_conv)
# plt.show()

In [ ]:
# Visualize predictions
# helper_utils.visualize_predictions(model_conv, dataloaders['val'], class_names, device, num_images=6)
# plt.show()

<a name='4-2'></a>
### 4.2 Fine-tuning the ConvNet

In this approach, we unfreeze all layers and train the entire network with a smaller learning rate.

**Task:**
1. Load a fresh pre-trained model
2. Modify the final layer
3. Set up optimizer for all parameters with a smaller learning rate
4. Train the model

**Reference:** [Transfer Learning Best Practices](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

In [ ]:
# Load a fresh pre-trained model
# YOUR CODE HERE
# model_ft = models.resnet18(weights='IMAGENET1K_V1')

# Modify final layer (all parameters will be trainable)
# num_ftrs = model_ft.fc.in_features
# model_ft.fc = nn.Linear(num_ftrs, len(class_names))
# model_ft = model_ft.to(device)

# Set up loss function and optimizer (for all parameters)
# criterion = nn.CrossEntropyLoss()
# optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
# exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

# Train the model
# model_ft, history_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=10)

In [ ]:
# Visualize training history
# helper_utils.visualize_training_history(history_ft)
# plt.show()

In [ ]:
# Visualize predictions
# helper_utils.visualize_predictions(model_ft, dataloaders['val'], class_names, device, num_images=6)
# plt.show()

<a name='5'></a>
## 5. Evaluation

Compare the performance of both approaches.

**Task:** Evaluate both models and compare their performance metrics.

In [ ]:
# Evaluate models on validation set
# YOUR CODE HERE
# Compare final validation accuracies, training times, etc.

# Print comparison
# print("Feature Extractor Approach:")
# print(f"  Best Val Accuracy: {max(history_conv['val_acc']):.4f}")
# print(f"  Final Val Accuracy: {history_conv['val_acc'][-1]:.4f}")
# print()
# print("Fine-tuning Approach:")
# print(f"  Best Val Accuracy: {max(history_ft['val_acc']):.4f}")
# print(f"  Final Val Accuracy: {history_ft['val_acc'][-1]:.4f}")

<a name='6'></a>
## 6. Inference on Custom Images

Test your trained model on custom images.

**Task:** Load a custom image, preprocess it, and make a prediction using your trained model.

**Reference:** [Image Preprocessing](https://pytorch.org/vision/stable/transforms.html)

In [ ]:
# Make prediction on a custom image
# img_path = 'path/to/your/image.jpg'

# YOUR CODE HERE
# Use helper_utils.visualize_single_prediction or helper_utils.predict_single_image
# helper_utils.visualize_single_prediction(
#     model_ft,  # or model_conv
#     img_path,
#     data_transforms['val'],
#     class_names,
#     device
# )
# plt.show()

<a name='7'></a>
## 7. Deployment (ONNX)

Convert your trained model to ONNX format for deployment.

**Task:** 
1. Convert the PyTorch model to ONNX format
2. Load the ONNX model and perform inference

**Reference:** 
- [PyTorch to ONNX](https://docs.pytorch.org/tutorials/beginner/onnx/export_simple_model_to_onnx_tutorial.html)

In [ ]:
# Convert model to ONNX
# YOUR CODE HERE

# Set model to evaluation mode
# model_ft.eval()

# Create dummy input (batch_size=1, channels=3, height=224, width=224)
# dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export to ONNX
# onnx_path = 'model.onnx'
# torch.onnx.export(
#     model_ft,
#     dummy_input,
#     onnx_path,
#     input_names=['input'],
#     output_names=['output'],
#     dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
# )

# print(f"Model exported to {onnx_path}")

In [ ]:
# Load ONNX model and perform inference
# YOUR CODE HERE

# Load ONNX model
# ort_session = ort.InferenceSession(onnx_path)

# Prepare input (use validation transform)
# img_path = 'path/to/test/image.jpg'
# img = Image.open(img_path).convert('RGB')
# img_tensor = data_transforms['val'](img).unsqueeze(0)
# img_numpy = img_tensor.numpy()

# Run inference
# outputs = ort_session.run(None, {'input': img_numpy})
# predictions = np.array(outputs[0])
# pred_class_idx = np.argmax(predictions[0])
# pred_class = class_names[pred_class_idx]
# confidence = np.max(predictions[0])

# print(f"Predicted: {pred_class} (confidence: {confidence:.2%})")

# 🏆🎉 Congratulations on completing the Weekly Final Project! 🎉🏆

Fantastic job on finishing the Weekly Final Project! You’ve put your skills to the test and made it to the end. Take a moment to celebrate your hard work and dedication. Keep up the great work and continue your learning journey!